# Demo Backtest Viewer

Visualize outputs from `tools/demo_backtest.py`.

This notebook loads:
- `summary.json`
- `equity_curve.csv`
- `weights_history.csv`
- `orders_history.csv`

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Change this if your backtest output directory differs
BACKTEST_DIR = Path('../eval_results/backtest/demo_backtest')
assert BACKTEST_DIR.exists(), f'Backtest dir not found: {BACKTEST_DIR.resolve()}'

summary = json.loads((BACKTEST_DIR / 'summary.json').read_text())
equity = pd.read_csv(BACKTEST_DIR / 'equity_curve.csv')
weights = pd.read_csv(BACKTEST_DIR / 'weights_history.csv')
orders = pd.read_csv(BACKTEST_DIR / 'orders_history.csv')

equity['trade_date'] = pd.to_datetime(equity['trade_date'])
equity['next_date'] = pd.to_datetime(equity['next_date'])
weights['trade_date'] = pd.to_datetime(weights['trade_date'])
if 'trade_date' in orders.columns:
    orders['trade_date'] = pd.to_datetime(orders['trade_date'])

summary

In [ ]:
summary_df = pd.DataFrame([summary]).T
summary_df.columns = ['value']
summary_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
equity.plot(x='trade_date', y='nav', ax=ax, legend=False, title='Equity Curve (NAV)')
ax.set_xlabel('trade_date')
ax.set_ylabel('nav')
plt.tight_layout()
plt.show()

In [ ]:
eq = equity[['trade_date', 'nav']].copy()
eq['running_max'] = eq['nav'].cummax()
eq['drawdown'] = eq['nav'] / eq['running_max'] - 1.0

fig, ax = plt.subplots(figsize=(10, 3))
eq.plot(x='trade_date', y='drawdown', ax=ax, legend=False, title='Drawdown')
ax.set_xlabel('trade_date')
ax.set_ylabel('drawdown')
plt.tight_layout()
plt.show()

eq[['trade_date', 'drawdown']].sort_values('drawdown').head(5)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
equity.plot(x='trade_date', y='portfolio_return', ax=ax, label='portfolio_return', alpha=0.85)
equity.plot(x='trade_date', y='benchmark_return', ax=ax, label='benchmark_return', alpha=0.85)
ax.set_title('Periodic Returns')
ax.set_xlabel('trade_date')
ax.set_ylabel('return')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
equity.plot(x='trade_date', y='turnover', ax=ax, legend=False, title='Turnover by Rebalance')
ax.set_xlabel('trade_date')
ax.set_ylabel('turnover')
plt.tight_layout()
plt.show()

equity[['trade_date', 'turnover', 'cost']].tail(10)

In [ ]:
# Top average weights across backtest period
weight_cols = [c for c in weights.columns if c != 'trade_date']
avg_weights = weights[weight_cols].mean().sort_values(ascending=False)
avg_weights.head(20)

In [ ]:
top_avg = avg_weights.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
top_avg.plot(kind='barh', ax=ax, title='Top 20 Average Weights')
ax.set_xlabel('average weight')
plt.tight_layout()
plt.show()

In [ ]:
orders.head(20)

In [ ]:
order_stats = {
    'total_orders': int(len(orders)),
    'total_estimated_cost': float(orders['estimated_cost'].sum()) if 'estimated_cost' in orders.columns else None,
    'buy_orders': int((orders['action'] == 'BUY').sum()) if 'action' in orders.columns else None,
    'sell_orders': int((orders['action'] == 'SELL').sum()) if 'action' in orders.columns else None,
}
order_stats